# 01b · Filters & Shapes `[local]`

**Goal:** understand **convolution**, the single most important operation in computer vision (CNNs are built from it), then use filters, edges, thresholds, morphology and contours to find and measure objects.

| Section | What you'll do |
|---|---|
| 🧠 Intuition | Convolution as a sliding stencil; edges as sudden changes |
| 📐 Math | Convolution formula, common kernels, gradients, Otsu's threshold |
| 🔨 From scratch | `convolve2d`, Otsu, erosion/dilation in NumPy, checked against OpenCV |
| 🚀 OpenCV | Blurs, Canny, thresholds, morphology, contours, Hough, watershed, GrabCut |
| 📊 Evaluate | Denoising PSNR table, coin-count error, shape-classifier accuracy, scanner corner error |
| 🛠 Projects | Photoshop filters · coin counter · shape detector · **document scanner** |

In [ ]:
# ── Standard first cell ──
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents, *pathlib.Path("/kaggle/input").glob("*")]:
    if (_p / "cvlib").is_dir():
        sys.path.insert(0, str(_p)); break
from cvlib.env import setup
env = setup()

import time
import cv2
import numpy as np
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view
from skimage import data as skdata
from cvlib.data import opencv_sample
from cvlib.viz import show

rng = np.random.default_rng(0)

---
# 🧠 1 · Intuition

### Convolution = a sliding stencil
Imagine a small **stencil** (say 3×3) with a number in each hole. Place it over the image. Multiply each pixel under the stencil by the number in its hole, add everything up, and write that **one number** into the output. Slide one pixel over and repeat.

The numbers in the stencil (the **kernel**) decide what happens:
- all equal → each pixel becomes the **average** of its neighbors → **blur**
- big positive center, negatives around it → exaggerate differences → **sharpen**
- negatives on the left, positives on the right → big response where brightness **jumps** left-to-right → **vertical edge detector**

**Why this matters for deep learning:** a CNN is a stack of convolutions whose kernel numbers are **learned** instead of hand-picked. Everything you build here is what a CNN's first layer learns by itself (module 03).

### Edges = where brightness changes suddenly
An object's outline is where the pixel values jump. Measure "how fast brightness changes" (the **gradient**) and you've found outlines.

### Threshold → shapes → measurements
Classic object finding is a 3-step recipe you'll use constantly:
1. **Separate** object from background (threshold → black/white mask)
2. **Clean** the mask (morphology: remove specks, fill holes)
3. **Find and measure** each blob (contours: area, perimeter, box, corners)

---
# 📐 2 · The math

### Convolution
For an image $I$ and a $k\times k$ kernel $K$ (with $r = \lfloor k/2 \rfloor$):

$$O(y, x) = \sum_{i=-r}^{r}\sum_{j=-r}^{r} K(i, j)\; I(y + i,\; x + j)$$

Strictly, that formula is **cross-correlation**. True convolution flips the kernel first: $K(-i, -j)$. For symmetric kernels (blur) they're identical. OpenCV's `filter2D` and PyTorch's `Conv2d` both actually compute **correlation**, and everyone still calls it convolution.

**Borders:** near the edge the stencil hangs off the image. We **pad** the image first (OpenCV's default reflects pixels: `dcb|abcd|cba`).

### Kernels you should recognize

| Name | Kernel | Effect |
|---|---|---|
| Box blur | $\frac{1}{9}\begin{bmatrix}1&1&1\\1&1&1\\1&1&1\end{bmatrix}$ | average |
| Gaussian | $\frac{1}{16}\begin{bmatrix}1&2&1\\2&4&2\\1&2&1\end{bmatrix}$ | smooth average, center counts most |
| Sharpen | $\begin{bmatrix}0&-1&0\\-1&5&-1\\0&-1&0\end{bmatrix}$ | original + (original − blur) |
| Sobel x | $\begin{bmatrix}-1&0&1\\-2&0&2\\-1&0&1\end{bmatrix}$ | left→right change (vertical edges) |
| Laplacian | $\begin{bmatrix}0&1&0\\1&-4&1\\0&1&0\end{bmatrix}$ | change in all directions |

Notice: **blur kernels sum to 1** (keep brightness), **edge kernels sum to 0** (flat areas → 0).

### Gradient magnitude and direction
With $G_x$ = Sobel-x response and $G_y$ = Sobel-y response:

$$|G| = \sqrt{G_x^2 + G_y^2} \qquad \theta = \arctan\!\left(\frac{G_y}{G_x}\right)$$

$|G|$ = edge strength, $\theta$ = which way the edge faces.

### Otsu's threshold
Try every threshold $t$ from 0 to 255. Each splits the pixels into "dark" and "bright" groups. Pick the $t$ that makes the two groups **as far apart as possible**:

$$t^* = \arg\max_t \; w_0(t)\, w_1(t)\, \big(\mu_0(t) - \mu_1(t)\big)^2$$

$w_0, w_1$ = fraction of pixels in each group, $\mu_0, \mu_1$ = their mean brightness.

---
# 🔨 3 · Build it from scratch

### 3.1 · `convolve2d`: two versions

First a slow, obvious loop (read this one), then a fast vectorized one (use this one).

In [ ]:
def convolve2d_loop(img, kernel):
    """Readable version: one output pixel at a time. Correlation, like OpenCV/PyTorch."""
    k = kernel.shape[0]; r = k // 2
    padded = np.pad(img.astype(np.float32), r, mode="reflect")     # reflect = OpenCV's default border
    out = np.zeros(img.shape, np.float32)
    for y in range(img.shape[0]):
        for x in range(img.shape[1]):
            window = padded[y:y + k, x:x + k]                        # the pixels under the stencil
            out[y, x] = np.sum(window * kernel)                      # multiply and add
    return out

def convolve2d(img, kernel):
    """Fast version: build all windows at once, then one big multiply-and-sum."""
    kh, kw = kernel.shape
    padded = np.pad(img.astype(np.float32), ((kh // 2, kh // 2), (kw // 2, kw // 2)), mode="reflect")
    windows = sliding_window_view(padded, (kh, kw))                 # shape (H, W, kh, kw): no copying!
    return np.einsum("hwij,ij->hw", windows, kernel.astype(np.float32))

gray = cv2.cvtColor(cv2.imread(str(opencv_sample("building.jpg"))), cv2.COLOR_BGR2GRAY)
small = gray[:120, :120]
k_sharpen = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], np.float32)

t0 = time.perf_counter(); a = convolve2d_loop(small, k_sharpen); t_loop = time.perf_counter() - t0
t0 = time.perf_counter(); b = convolve2d(small, k_sharpen); t_fast = time.perf_counter() - t0
c = cv2.filter2D(small.astype(np.float32), -1, k_sharpen)
print(f"loop vs fast max diff : {np.abs(a - b).max():.4f}")
print(f"fast vs OpenCV max diff: {np.abs(b - c).max():.4f}")
print(f"loop {t_loop*1000:.0f} ms | vectorized {t_fast*1000:.1f} ms  → {t_loop / t_fast:.0f}× faster on a 120×120 crop")

Both match OpenCV exactly. **Lesson:** never loop over pixels in Python. Express the operation as array math.

### 3.2 · Gradients by hand: Sobel and edge magnitude

In [ ]:
sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], np.float32)
sobel_y = sobel_x.T
gx = convolve2d(gray, sobel_x)
gy = convolve2d(gray, sobel_y)
mag = np.sqrt(gx ** 2 + gy ** 2)

def to_u8(x):
    """Stretch any float array to 0..255 for display."""
    x = np.abs(x); return (255 * x / x.max()).astype(np.uint8)

show([gray, to_u8(gx), to_u8(gy), to_u8(mag)], ["input", "|Gx|: vertical edges", "|Gy|: horizontal edges", "magnitude: all edges"], cols=4, size=3)

### 3.3 · Otsu's threshold by hand

In [ ]:
def otsu_threshold(gray):
    hist = np.bincount(gray.ravel(), minlength=256).astype(np.float64)
    p = hist / hist.sum()                       # probability of each brightness
    omega = np.cumsum(p)                        # w0(t): fraction of pixels ≤ t
    mu = np.cumsum(p * np.arange(256))          # cumulative mean
    mu_total = mu[-1]
    # between-class variance for every t at once (the formula above, rearranged)
    with np.errstate(divide="ignore", invalid="ignore"):
        sigma_b = (mu_total * omega - mu) ** 2 / (omega * (1 - omega))
    return int(np.nanargmax(sigma_b))

page = skdata.page()                            # scanned text with uneven lighting
t_mine = otsu_threshold(page)
t_cv, otsu_cv = cv2.threshold(page, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
print(f"my Otsu threshold: {t_mine} | OpenCV: {t_cv:.0f}")

### 3.4 · Erosion and dilation by hand

Morphology works on black/white masks with a small shape called a **structuring element**:
- **Erosion** = a pixel stays white only if **all** neighbors are white (sliding **minimum**) → shrinks blobs, deletes specks
- **Dilation** = a pixel becomes white if **any** neighbor is white (sliding **maximum**) → grows blobs, fills gaps

In [ ]:
def erode(mask, k=3):
    padded = np.pad(mask, k // 2, mode="constant", constant_values=255)
    return sliding_window_view(padded, (k, k)).min(axis=(2, 3))

def dilate(mask, k=3):
    padded = np.pad(mask, k // 2, mode="constant", constant_values=0)
    return sliding_window_view(padded, (k, k)).max(axis=(2, 3))

text_mask = 255 - otsu_cv                       # white text on black
kernel = np.ones((3, 3), np.uint8)
print("erode matches OpenCV :", np.array_equal(erode(text_mask), cv2.erode(text_mask, kernel)))
print("dilate matches OpenCV:", np.array_equal(dilate(text_mask), cv2.dilate(text_mask, kernel)))
crop = (slice(0, 90), slice(0, 200))
show([text_mask[crop], erode(text_mask)[crop], dilate(text_mask)[crop]], ["mask", "eroded (thinner)", "dilated (bolder)"], cols=3, size=4)

---
# 🛠 Project 1 · Photoshop filters with your own `convolve2d`

Every classic filter is just a different kernel. Apply each one to every color channel.

In [ ]:
def apply_kernel_color(img, kernel):
    return np.dstack([np.clip(convolve2d(img[:, :, c], kernel), 0, 255) for c in range(3)]).astype(np.uint8)

def gaussian_kernel(size=9, sigma=2.0):
    ax = np.arange(size) - size // 2
    g = np.exp(-(ax ** 2) / (2 * sigma ** 2))
    k = np.outer(g, g)
    return k / k.sum()                                    # sums to 1 → brightness preserved

kernels = {
    "blur (Gaussian 9×9)": gaussian_kernel(9, 2.0),
    "sharpen": np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], np.float32),
    "emboss": np.array([[-2, -1, 0], [-1, 1, 1], [0, 1, 2]], np.float32),
    "motion blur (→)": np.pad(np.full((1, 15), 1 / 15, np.float32), ((7, 7), (0, 0))),   # one horizontal line of 1/15
    "outline (Laplacian)": np.array([[-1, -1, -1], [-1, 8, -1], [-1, -1, -1]], np.float32),
}
photo = cv2.resize(cv2.imread(str(opencv_sample("baboon.jpg"))), (256, 256), interpolation=cv2.INTER_AREA)
outs = [apply_kernel_color(photo, k) for k in kernels.values()]
show([photo] + outs, ["original"] + list(kernels), cols=3, size=3.3)

In [ ]:
# Check every filter against OpenCV (filter2D = correlation with reflect border, same as ours)
for name, k in kernels.items():
    ref = np.clip(cv2.filter2D(photo.astype(np.float32), -1, k.astype(np.float32)), 0, 255).astype(np.uint8)
    mine = apply_kernel_color(photo, k)
    print(f"{name:22s} max diff vs OpenCV: {np.abs(ref.astype(int) - mine.astype(int)).max()}")

---
# 🚀 4 · The OpenCV way

### 4.1 · Three blurs, three personalities

| Filter | How | Strength |
|---|---|---|
| `GaussianBlur` | weighted average | general smoothing, fast |
| `medianBlur` | **median** of the window | destroys salt-and-pepper specks, keeps edges |
| `bilateralFilter` | average only **similar-colored** neighbors | smooths skin/flat areas, keeps edges sharp |

In [ ]:
clean = cv2.cvtColor(photo, cv2.COLOR_BGR2GRAY)
gauss_noisy = np.clip(clean + rng.normal(0, 20, clean.shape), 0, 255).astype(np.uint8)
sp_noisy = clean.copy()
m = rng.random(clean.shape)
sp_noisy[m < 0.05] = 0; sp_noisy[m > 0.95] = 255                  # 10% salt-and-pepper

filters = {
    "Gaussian 5": lambda x: cv2.GaussianBlur(x, (5, 5), 0),
    "Median 5": lambda x: cv2.medianBlur(x, 5),
    "Bilateral": lambda x: cv2.bilateralFilter(x, 9, 50, 9),
}
show([sp_noisy] + [f(sp_noisy) for f in filters.values()], ["salt & pepper noise"] + list(filters), cols=4, size=3)

### 4.2 · Canny: the standard edge detector

Canny = 4 steps on top of what you built:
1. Gaussian blur (reduce noise)
2. Sobel gradients (magnitude + direction)
3. **Non-maximum suppression**: thin thick edges to 1 pixel by keeping only the peak across the edge
4. **Hysteresis**: pixels above `high` are edges; pixels between `low` and `high` count only if connected to a strong edge

In [ ]:
edges_loose = cv2.Canny(gray, 30, 80)
edges_mid = cv2.Canny(gray, 50, 150)
edges_strict = cv2.Canny(gray, 150, 300)
show([to_u8(mag), edges_loose, edges_mid, edges_strict], ["your Sobel magnitude", "Canny 30/80", "Canny 50/150", "Canny 150/300"], cols=4, size=3)

### 4.3 · Thresholding: global, Otsu, adaptive

A single global threshold fails when lighting varies across the image. **Adaptive thresholding** computes a different threshold for each pixel from its neighborhood.

In [ ]:
_, fixed = cv2.threshold(page, 127, 255, cv2.THRESH_BINARY)
adaptive = cv2.adaptiveThreshold(page, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 25, 10)
show([page, fixed, otsu_cv, adaptive], ["scanned page (uneven light)", "fixed t=127", f"Otsu t={t_cv:.0f}", "adaptive (block 25)"], cols=2, size=5)

Adaptive wins on documents: the dark bottom-left is lost by the global methods. This is exactly the binarization step used before OCR in module 16.

### 4.4 · Morphology: opening, closing and friends

| Operation | = | Use |
|---|---|---|
| **Opening** | erode → dilate | remove small white specks |
| **Closing** | dilate → erode | fill small black holes and gaps |
| **Gradient** | dilate − erode | outlines |
| **Top-hat** | image − opening | bright small things on uneven background |
| **Black-hat** | closing − image | dark small things (e.g. text) on uneven background |

In [ ]:
noisy_mask = text_mask.copy()
noisy_mask[rng.random(noisy_mask.shape) < 0.02] = 255                # sprinkle white specks
k3 = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
show([noisy_mask[crop], cv2.morphologyEx(noisy_mask, cv2.MORPH_OPEN, k3)[crop],
      cv2.morphologyEx(text_mask, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (9, 3)))[crop],
      cv2.morphologyEx(text_mask, cv2.MORPH_GRADIENT, k3)[crop]],
     ["specks added", "opening removes specks", "closing (9×3) joins letters into words", "gradient = outlines"], cols=2, size=5)

### 4.5 · Contours: finding and measuring shapes

`cv2.findContours` traces the outline of every white blob in a mask. From each contour you can measure:

| Function | Gives |
|---|---|
| `cv2.contourArea(c)` | area in pixels |
| `cv2.arcLength(c, True)` | perimeter |
| `cv2.boundingRect(c)` | upright box `(x, y, w, h)` |
| `cv2.minAreaRect(c)` | tightest **rotated** box |
| `cv2.approxPolyDP(c, eps, True)` | simplified polygon → **count the corners** |
| `cv2.moments(c)` | centroid: `(m10/m00, m01/m00)` |
| `cv2.convexHull(c)` | "rubber band" around the shape |

**Circularity** $= \dfrac{4\pi \cdot \text{area}}{\text{perimeter}^2}$ is 1.0 for a perfect circle and smaller for everything else.

`RETR_EXTERNAL` returns only outer outlines; `RETR_TREE` also returns holes, with a parent/child hierarchy.

## 🛠 Project 2 · Shape detector

We generate a test image with **known** shapes, so we can measure accuracy honestly instead of eyeballing.

In [ ]:
SHAPES = ["triangle", "square", "rectangle", "pentagon", "hexagon", "circle"]

def regular_polygon(cx, cy, r, n, angle):
    a = angle + np.arange(n) * 2 * np.pi / n
    return np.stack([cx + r * np.cos(a), cy + r * np.sin(a)], 1).astype(np.int32)

def make_shapes_image(n=24, size=(600, 800), seed=1):
    r_ = np.random.default_rng(seed)
    img = np.full((*size, 3), 30, np.uint8)
    truth, placed = [], []
    while len(truth) < n:
        cx, cy, rad = r_.integers(60, size[1] - 60), r_.integers(60, size[0] - 60), r_.integers(30, 50)
        if any(np.hypot(cx - px, cy - py) < rad + pr + 10 for px, py, pr in placed):
            continue                                                  # avoid overlaps
        kind = SHAPES[r_.integers(len(SHAPES))]
        color = tuple(int(v) for v in r_.integers(80, 256, 3))
        ang = r_.uniform(0, np.pi)
        if kind == "circle":
            cv2.circle(img, (int(cx), int(cy)), int(rad), color, -1)
        elif kind == "rectangle":
            box = cv2.boxPoints(((float(cx), float(cy)), (float(rad * 2.0), float(rad * 1.0)), float(np.degrees(ang))))
            cv2.fillPoly(img, [box.astype(np.int32)], color)
        else:
            nv = {"triangle": 3, "square": 4, "pentagon": 5, "hexagon": 6}[kind]
            cv2.fillPoly(img, [regular_polygon(cx, cy, rad, nv, ang)], color)
        truth.append((kind, int(cx), int(cy))); placed.append((cx, cy, rad))
    img = np.clip(img + r_.normal(0, 6, img.shape), 0, 255).astype(np.uint8)   # a bit of sensor noise
    return img, truth

shapes_img, truth = make_shapes_image()
show(shapes_img, "test image: 24 random shapes with known labels", size=7)

In [ ]:
def classify_contour(c):
    peri = cv2.arcLength(c, True)
    approx = cv2.approxPolyDP(c, 0.03 * peri, True)          # epsilon = 3% of perimeter
    n = len(approx)
    circularity = 4 * np.pi * cv2.contourArea(c) / peri ** 2
    if n == 3:
        return "triangle"
    if n == 4:
        (_, _), (w, h), _ = cv2.minAreaRect(c)                # rotation-proof width/height
        return "square" if 0.85 < w / h < 1.18 else "rectangle"
    if n == 5:
        return "pentagon"
    if n == 6 and circularity < 0.88:
        return "hexagon"
    return "circle"

def detect_shapes(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(cv2.GaussianBlur(gray, (5, 5), 0), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    found = []
    for c in contours:
        if cv2.contourArea(c) < 200:
            continue
        M = cv2.moments(c)
        found.append((classify_contour(c), int(M["m10"] / M["m00"]), int(M["m01"] / M["m00"]), c))
    return found

found = detect_shapes(shapes_img)
vis = shapes_img.copy()
for label, cx, cy, c in found:
    cv2.drawContours(vis, [c], -1, (0, 255, 0), 2)
    cv2.putText(vis, label, (cx - 35, cy + 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 3, cv2.LINE_AA)
    cv2.putText(vis, label, (cx - 35, cy + 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
show(vis, "predicted labels", size=7)

## 🛠 Project 3 · Coin counter

`skimage.data.coins()` has **24 coins** (4 rows of 6). The catch: **uneven lighting**, brighter at the top.

In [ ]:
coins = skdata.coins()
coins_blur = cv2.GaussianBlur(coins, (5, 5), 0)

def count_blobs(mask, min_area=300):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return [c for c in contours if cv2.contourArea(c) >= min_area]

# Attempt A: global Otsu
_, mask_a = cv2.threshold(coins_blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# Attempt B: remove the uneven background first with a top-hat (image − opening with a big disk), then Otsu
big_disk = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (81, 81))       # bigger than any coin
flat = cv2.morphologyEx(coins_blur, cv2.MORPH_TOPHAT, big_disk)
_, mask_b = cv2.threshold(flat, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
mask_b = cv2.morphologyEx(mask_b, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)))

# Attempt C: Hough circle transform (works on gradients, not on a mask)
circles = cv2.HoughCircles(coins_blur, cv2.HOUGH_GRADIENT, dp=1, minDist=20, param1=100, param2=25, minRadius=8, maxRadius=40)
circles = np.round(circles[0]).astype(int) if circles is not None else np.empty((0, 3), int)

blobs_a, blobs_b = count_blobs(mask_a), count_blobs(mask_b)
vis_c = cv2.cvtColor(coins, cv2.COLOR_GRAY2BGR)
for x, y, r in circles:
    cv2.circle(vis_c, (x, y), r, (0, 0, 255), 2)
show([mask_a, flat, mask_b, vis_c],
     [f"A: global Otsu → {len(blobs_a)} blobs", "top-hat removes the lighting gradient", f"B: top-hat + Otsu → {len(blobs_b)} blobs", f"C: Hough circles → {len(circles)}"],
     cols=2, size=5)

**Why A fails:** the bright background at the top is above Otsu's threshold, so coins merge with it into one huge blob. **Fix the lighting before thresholding.** That lesson applies to documents, microscopy and factory inspection alike.

### 4.6 · Hough transform: finding lines and circles

Each edge pixel **votes** for every line (or circle) that could pass through it. Lines with lots of votes are real. That's how Attempt C above found the coins.

In [ ]:
sudoku = cv2.imread(str(opencv_sample("sudoku.png")))
sud_edges = cv2.Canny(cv2.cvtColor(sudoku, cv2.COLOR_BGR2GRAY), 50, 150)
lines = cv2.HoughLinesP(sud_edges, rho=1, theta=np.pi / 180, threshold=120, minLineLength=150, maxLineGap=10)
vis_l = sudoku.copy()
for x1, y1, x2, y2 in lines.reshape(-1, 4):                         # shape (N, 4) in OpenCV 5, (N, 1, 4) in 4.x
    cv2.line(vis_l, (x1, y1), (x2, y2), (0, 0, 255), 2)

smarties = cv2.imread(str(opencv_sample("smarties.png")))
sm_gray = cv2.medianBlur(cv2.cvtColor(smarties, cv2.COLOR_BGR2GRAY), 5)
sm_circles = cv2.HoughCircles(sm_gray, cv2.HOUGH_GRADIENT, dp=1, minDist=20, param1=100, param2=30, minRadius=10, maxRadius=40)
vis_s = smarties.copy()
for x, y, r in np.round(sm_circles[0]).astype(int):
    cv2.circle(vis_s, (x, y), r, (0, 255, 0), 2)
show([vis_l, vis_s], [f"HoughLinesP: {len(lines)} segments", f"HoughCircles: {sm_circles.shape[1]} candies"], size=5)

### 4.7 · Watershed: splitting objects that touch

When blobs touch, contours see **one** object. **Watershed** treats brightness as a landscape and floods it from markers, building walls where floods meet.

Recipe: **distance transform** (how far each pixel is from the background) → **local peaks** = object centers → markers → `cv2.watershed`.

A local peak is a pixel that equals the maximum of its neighborhood. We find them by comparing the image with a dilated copy of itself (dilation = sliding max).

In [ ]:
touching = np.zeros((300, 400), np.uint8)
for (x, y, r) in [(100, 150, 60), (190, 150, 55), (270, 120, 45), (300, 200, 50)]:
    cv2.circle(touching, (x, y), r, 255, -1)

dist = cv2.distanceTransform(touching, cv2.DIST_L2, 5)
is_peak = (dist == cv2.dilate(dist, np.ones((25, 25), np.uint8))) & (dist > 0.3 * dist.max())
peaks = cv2.dilate(is_peak.astype(np.uint8) * 255, np.ones((5, 5), np.uint8))   # grow each peak to a small seed
n_markers, markers = cv2.connectedComponents(peaks)
markers = markers + 1                                              # background = 1, objects = 2..n
unknown = cv2.subtract(cv2.dilate(touching, np.ones((3, 3), np.uint8), iterations=3), peaks)
markers[unknown == 255] = 0                                         # 0 = "let watershed decide"
ws = cv2.watershed(cv2.cvtColor(touching, cv2.COLOR_GRAY2BGR), markers)

vis_ws = cv2.cvtColor(touching, cv2.COLOR_GRAY2BGR)
vis_ws[ws == -1] = (0, 0, 255)                                      # -1 = boundaries
print("contours see:", len(count_blobs(touching)), "object(s) | watershed sees:", len(np.unique(ws)) - 2, "objects")
show([touching, to_u8(dist), peaks, vis_ws], ["touching blobs", "distance transform", "local peaks = markers", "watershed boundaries (red)"], cols=4, size=3)

### 4.8 · GrabCut: cut out an object from a rough box

Give GrabCut a rectangle around the object. It models foreground and background colors and iteratively decides each pixel. It's the ancestor of "remove background" buttons, and of SAM in module 07.

In [ ]:
cat = cv2.cvtColor(skdata.chelsea(), cv2.COLOR_RGB2BGR)
gc_mask = np.zeros(cat.shape[:2], np.uint8)
rect = (60, 20, 300, 270)                                           # x, y, w, h around the cat
bgd, fgd = np.zeros((1, 65), np.float64), np.zeros((1, 65), np.float64)
cv2.grabCut(cat, gc_mask, rect, bgd, fgd, 5, cv2.GC_INIT_WITH_RECT)
fg = np.where((gc_mask == cv2.GC_FGD) | (gc_mask == cv2.GC_PR_FGD), 1, 0).astype(np.uint8)
vis_r = cat.copy(); cv2.rectangle(vis_r, rect[:2], (rect[0] + rect[2], rect[1] + rect[3]), (0, 255, 0), 2)
show([vis_r, cat * fg[:, :, None]], ["input box", "GrabCut result"], size=4)

---
# 🛠 Project 4 · Automatic document scanner

The workflow behind every "scan with your phone" app, and the first step of the OCR pipeline in module 16:

1. grayscale → blur → **Canny edges** → dilate (close small gaps)
2. **contours**, sorted by area, largest first
3. first one that simplifies to **4 corners** (`approxPolyDP`) = the page
4. **order** the corners (top-left, top-right, bottom-right, bottom-left)
5. **perspective warp** to a flat rectangle (from 01a)
6. **adaptive threshold** for a clean black-and-white scan

To evaluate it honestly, we generate a fake phone photo where we **know** the true corners.

In [ ]:
def make_page(w=600, h=800, seed=0):
    r_ = np.random.default_rng(seed)
    page = np.full((h, w, 3), 245, np.uint8)
    cv2.putText(page, "INVOICE #2024-117", (40, 80), cv2.FONT_HERSHEY_DUPLEX, 1.2, (20, 20, 20), 2, cv2.LINE_AA)
    words = ["total", "amount", "date", "item", "qty", "price", "tax", "due", "paid", "ref", "note"]
    for i in range(14):
        cv2.putText(page, " ".join(r_.choice(words, 5)), (40, 150 + 40 * i), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (40, 40, 40), 1, cv2.LINE_AA)
    cv2.rectangle(page, (40, 710), (560, 780), (60, 60, 60), 2)
    cv2.putText(page, "TOTAL DUE: $1,284.50", (60, 755), cv2.FONT_HERSHEY_DUPLEX, 0.9, (10, 10, 10), 2, cv2.LINE_AA)
    return page

def make_phone_photo(page, corners, size=(900, 1200), seed=0):
    """Paste `page` onto a wood-like table at `corners` (TL, TR, BR, BL), add shadow, noise, blur."""
    r_ = np.random.default_rng(seed)
    H, W = size
    grain = cv2.GaussianBlur(cv2.resize(r_.normal(0, 1, (H // 8, W)).astype(np.float32), (W, H)), (0, 0), 3)
    table = np.stack([60 + 15 * grain, 90 + 20 * grain, 130 + 25 * grain], -1).clip(0, 255).astype(np.uint8)
    h, w = page.shape[:2]
    M = cv2.getPerspectiveTransform(np.float32([[0, 0], [w, 0], [w, h], [0, h]]), np.float32(corners))
    warped = cv2.warpPerspective(page, M, (W, H))
    inside = cv2.warpPerspective(np.full((h, w), 255, np.uint8), M, (W, H)) > 0
    photo = np.where(inside[..., None], warped, table).astype(np.float32)
    light = np.linspace(0.65, 1.1, W)[None, :, None] * np.linspace(1.0, 0.85, H)[:, None, None]   # uneven light
    photo = np.clip(photo * light + r_.normal(0, 4, photo.shape), 0, 255).astype(np.uint8)
    return cv2.GaussianBlur(photo, (3, 3), 0)

true_corners = np.float32([[330, 120], [860, 190], [900, 820], [250, 760]])
photo_doc = make_phone_photo(make_page(), true_corners)
show(photo_doc, "fake phone photo of an invoice", size=6)

In [ ]:
def order_corners(pts):
    """Return corners as TL, TR, BR, BL. TL has the smallest x+y, BR the largest; TR has the smallest y−x."""
    pts = pts.reshape(4, 2).astype(np.float32)
    s, d = pts.sum(1), np.diff(pts, axis=1).ravel()
    return np.float32([pts[np.argmin(s)], pts[np.argmin(d)], pts[np.argmax(s)], pts[np.argmax(d)]])

def find_document(img, blur=5):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    edges = cv2.dilate(cv2.Canny(cv2.GaussianBlur(gray, (blur, blur), 0), 50, 150), np.ones((3, 3), np.uint8))
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for c in sorted(contours, key=cv2.contourArea, reverse=True)[:5]:
        approx = cv2.approxPolyDP(c, 0.02 * cv2.arcLength(c, True), True)
        if len(approx) == 4:
            return order_corners(approx), edges
    return None, edges

def scan(img, out_w=600, out_h=800):
    corners, edges = find_document(img)
    if corners is None:
        raise ValueError("no 4-corner page found")
    target = np.float32([[0, 0], [out_w - 1, 0], [out_w - 1, out_h - 1], [0, out_h - 1]])
    flat = cv2.warpPerspective(img, cv2.getPerspectiveTransform(corners, target), (out_w, out_h))
    bw = cv2.adaptiveThreshold(cv2.cvtColor(flat, cv2.COLOR_BGR2GRAY), 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 21, 10)
    return corners, edges, flat, bw

corners, edges_doc, flat_doc, bw_doc = scan(photo_doc)
vis_d = photo_doc.copy()
cv2.polylines(vis_d, [corners.astype(np.int32)], True, (0, 255, 0), 4)
show([edges_doc, vis_d, flat_doc, bw_doc], ["1. edges", "2–4. largest 4-corner contour", "5. flattened", "6. adaptive threshold"], cols=4, size=3.5)

It works on a real photo too, with no changes:

In [ ]:
s_corners, _, s_flat, s_bw = scan(sudoku, 450, 450)
vis_sd = sudoku.copy(); cv2.polylines(vis_sd, [s_corners.astype(np.int32)], True, (0, 255, 0), 3)
show([vis_sd, s_flat, s_bw], ["found automatically", "flattened", "binarized"], cols=3, size=3.8)

---
# 📊 5 · Evaluate

Numbers, not impressions.

### 5.1 · Denoising: which filter for which noise?

In [ ]:
def psnr(a, b):
    mse = np.mean((a.astype(np.float64) - b.astype(np.float64)) ** 2)
    return 10 * np.log10(255 ** 2 / mse)

print(f"{'filter':<12}{'Gaussian noise':>16}{'salt & pepper':>16}   (PSNR dB vs clean, higher = better)")
print(f"{'(none)':<12}{psnr(clean, gauss_noisy):>16.2f}{psnr(clean, sp_noisy):>16.2f}")
for name, f in filters.items():
    print(f"{name:<12}{psnr(clean, f(gauss_noisy)):>16.2f}{psnr(clean, f(sp_noisy)):>16.2f}")

**Takeaway:** match the filter to the noise. **Median** is far ahead on salt-and-pepper, because one extreme pixel can't move a median. Gaussian-type filters do better on Gaussian noise.

### 5.2 · Coin counter: count error

In [ ]:
TRUE_COINS = 24
for name, n in [("A: global Otsu", len(blobs_a)), ("B: top-hat + Otsu", len(blobs_b)), ("C: Hough circles", len(circles))]:
    print(f"{name:<20} counted {n:>2} | error {n - TRUE_COINS:+d}")

A count can be right for the wrong reasons (one miss + one false alarm = 0 error). A stricter test matches each detection to a real coin, which you'll formalize as **precision and recall** in module 06.

### 5.3 · Shape detector: accuracy against the known labels

Match each prediction to the nearest true shape center, then compare labels.

In [ ]:
correct, confusions = 0, []
for label, cx, cy, _ in found:
    t_label, tx, ty = min(truth, key=lambda t: np.hypot(t[1] - cx, t[2] - cy))
    correct += label == t_label
    if label != t_label:
        confusions.append(f"{t_label} → predicted {label}")
print(f"found {len(found)} / {len(truth)} shapes | label accuracy: {correct}/{len(found)} = {correct / len(found):.0%}")
print("mistakes:", confusions or "none")

### 5.4 · Document scanner: corner error and robustness

Corner error = average distance in pixels between found and true corners. Then the real test: **does it still work as conditions get harder?**

In [ ]:
err = np.linalg.norm(corners - order_corners(true_corners), axis=1).mean()
print(f"mean corner error on the default photo: {err:.2f} px")

r_ = np.random.default_rng(42)
results = []
for trial in range(30):
    jitter = true_corners + r_.uniform(-80, 80, true_corners.shape).astype(np.float32)
    img_t = make_phone_photo(make_page(seed=trial), jitter, seed=trial)
    noise_level = [0, 30, 60][trial % 3]                                      # harder every 3rd photo
    img_t = np.clip(img_t + r_.normal(0, noise_level, img_t.shape), 0, 255).astype(np.uint8)
    c, _ = find_document(img_t)
    e = np.inf if c is None else np.linalg.norm(c - order_corners(jitter), axis=1).mean()
    results.append((noise_level, e))

for nl in [0, 30, 60]:
    errs = np.array([e for n, e in results if n == nl])
    ok = np.isfinite(errs) & (errs < 5)
    print(f"extra noise σ={nl:>2}: success (error < 5 px) {ok.sum():>2}/{len(errs)} | median error {np.median(errs[np.isfinite(errs)]) if np.isfinite(errs).any() else float('nan'):.2f} px")

It's solid up to moderate noise and **collapses at σ=60**. That's the kind of finding you only get from a stress test with a known answer, the pattern you'll use to evaluate every pipeline in this course. Experiment 4 fixes it.

---
# 🧪 6 · Experiments: predict first, then run

**Experiment 1: kernel size.** Blur the baboon with Gaussian kernels of size 3, 9, 31. Predict which details disappear first.

In [ ]:
# Prediction: ...
show([cv2.GaussianBlur(photo, (k, k), 0) for k in (3, 9, 31)], ["3×3", "9×9", "31×31"], cols=3, size=3.3)

**Experiment 2: an edge kernel that sums to 1 instead of 0.** Add 1 to the Laplacian's center. What will the output look like?

In [ ]:
# Prediction: ...
lap = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], np.float32)
show([to_u8(convolve2d(clean, lap)), to_u8(convolve2d(clean, lap + np.array([[0, 0, 0], [0, 1, 0], [0, 0, 0]])))],
     ["sum = 0: only edges", "sum = 1: edges + original image"], size=4)

**Experiment 3: `approxPolyDP` epsilon.** The shape classifier uses 3% of the perimeter. What happens at 0.5% and 10%?

In [ ]:
# Prediction: ...
c0 = found[0][3]
for eps in [0.005, 0.03, 0.10]:
    print(f"epsilon {eps:>5.1%} of perimeter → {len(cv2.approxPolyDP(c0, eps * cv2.arcLength(c0, True), True))} corners (true: {found[0][0]})")

**Experiment 4: fix the scanner at σ=60.** The stress test failed at heavy noise. Look at the edges first: what went wrong? Then predict: will a 9×9 blur before Canny fix it? What about 21×21?

In [ ]:
# Prediction: ...
hard = np.clip(photo_doc + rng.normal(0, 60, photo_doc.shape), 0, 255).astype(np.uint8)
panels, titles = [hard], ["σ=60 noise"]
for blur in (5, 9, 21):
    c_hard, e_hard = find_document(hard, blur=blur)
    ok = c_hard is not None and np.linalg.norm(c_hard - order_corners(true_corners), axis=1).mean() < 5
    panels.append(e_hard); titles.append(f"blur {blur}×{blur}: {'found ✓' if ok else 'failed ✗'}")
show(panels, titles, cols=4, size=3.5)
# 5×5: noise creates thousands of tiny edges, which break up the page outline.
# 9×9: noise is smoothed away, and the page edge survives.
# 21×21: the page edge itself gets too soft for Canny's thresholds. Too much of a good thing.

---
# 📝 7 · Recap

**Cheat sheet**
- **Convolution** = slide a kernel, multiply, sum. OpenCV/PyTorch actually compute correlation (no flip).
- Blur kernels sum to 1; edge kernels sum to 0. Never loop over pixels in Python.
- **Gaussian** for general noise, **median** for salt-and-pepper, **bilateral** to smooth but keep edges.
- **Canny** = blur → Sobel → non-max suppression → hysteresis (low/high thresholds).
- **Otsu** picks a global threshold automatically; **adaptive** thresholding handles uneven light (documents!).
- Fix uneven lighting **before** thresholding (top-hat, CLAHE, or dividing by a heavy blur).
- Opening removes specks, closing fills gaps. Erode = sliding min, dilate = sliding max.
- Contours → area, perimeter, boxes, `approxPolyDP` corners, circularity, centroid.
- Hough = edge pixels vote for lines/circles. Watershed splits touching objects. GrabCut cuts out an object from a box.
- **Evaluate pipelines on synthetic data with a known answer** and stress-test them.

**Interview-style questions**
1. What's the difference between convolution and correlation? Which does `nn.Conv2d` compute?
2. Why does a blur kernel sum to 1 and an edge kernel to 0?
3. Median vs. Gaussian blur: when is each better, and why?
4. Walk through Canny's four steps. What do the two thresholds control?
5. Your global threshold fails on a photographed document. Give two fixes.
6. How would you count touching coins?
7. Design an automatic document scanner. Where could it break, and how would you test it?

**Next:** `01c_features_and_geometry.ipynb`, where you'll match the same object across two photos and stitch a panorama.